In [24]:
# Read Silver Delta Tables

df_customers = spark.read.table("silver_customers")
df_orders = spark.read.table("silver_orders")
df_order_items = spark.read.table("silver_order_items")
df_products = spark.read.table("silver_products")
df_returns = spark.read.table("silver_returns")
df_sales_targets = spark.read.table("silver_sales_targets")

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 26, Finished, Available, Finished, False)

In [45]:
from pyspark.sql.functions import col

gold_sales = (
    df_order_items
    .join(df_orders, "OrderID")
    .join(df_products, "ProductID")
    .select(
        col("OrderID"),
        col("OrderDate"),
        col("CustomerID"),
        col("SalesChannel"),
        col("ProductName"),
        col("Category"),
        col("Quantity"),
        col("UnitPrice"),
        (col("Quantity") * col("UnitPrice")).alias("Revenue")
    )
)

display(gold_sales)

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bf1f9f2a-5f53-4aff-ac3b-eb13c250f240)

In [46]:
gold_sales.write \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .format("delta") \
    .saveAsTable("gold_sales")

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 48, Finished, Available, Finished, False)

In [47]:
display(gold_sales)

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 49, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a2cecc8c-fed0-41d1-870a-be3082c2df0a)

In [30]:
from pyspark.sql.functions import sum

gold_sales_by_category = (
    gold_sales
    .groupBy("Category")
    .agg(
        sum("Revenue").alias("TotalRevenue")
    )
    .orderBy("TotalRevenue", ascending=False)
)

display(gold_sales_by_category)

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a896bb36-0072-4820-89c3-b457c6f477ee)

In [31]:
gold_sales_by_category.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_sales_by_category")

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 33, Finished, Available, Finished, False)

In [32]:
gold_sales = spark.read.table("gold_sales")

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 34, Finished, Available, Finished, False)

In [33]:
from pyspark.sql.functions import sum

gold_sales_with_customer = (
    gold_sales
    .join(df_customers, "CustomerID")
)

gold_sales_by_province = (
    gold_sales_with_customer
    .groupBy("Province")
    .agg(
        sum("Revenue").alias("TotalRevenue")
    )
    .orderBy("TotalRevenue", ascending=False)
)

display(gold_sales_by_province)

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b3e05954-8877-4b93-861b-600f9c30e23a)

In [34]:
gold_sales_by_province.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_sales_by_province")

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 36, Finished, Available, Finished, False)

In [35]:
from pyspark.sql.functions import sum, countDistinct

gold_customer_summary = (
    gold_sales_with_customer
    .groupBy("CustomerID", "CustomerName", "CustomerSegment", "Province")
    .agg(
        sum("Revenue").alias("TotalRevenue"),
        countDistinct("OrderID").alias("TotalOrders")
    )
    .orderBy("TotalRevenue", ascending=False)
)

display(gold_customer_summary)

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c48e327a-c5db-4342-8b73-9bdde298b09b)

In [36]:
gold_customer_summary.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_customer_summary")

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 38, Finished, Available, Finished, False)

In [37]:
from pyspark.sql.functions import sum, date_format

actual_sales_by_month = (
    gold_sales
    .withColumn("TargetMonth", date_format("OrderDate", "yyyy-MM"))
    .groupBy("TargetMonth")
    .agg(
        sum("Revenue").alias("ActualRevenue")
    )
)

gold_target_vs_actual = (
    actual_sales_by_month
    .join(df_sales_targets, "TargetMonth", "left")
)

display(gold_target_vs_actual)

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 350bb023-6d23-4a6f-b203-6bc0d772bdf9)

In [38]:
gold_target_vs_actual.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_target_vs_actual")

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 40, Finished, Available, Finished, False)

In [39]:
from pyspark.sql.functions import sum

gold_top_products = (
    gold_sales
    .groupBy("ProductName")
    .agg(
        sum("Revenue").alias("TotalRevenue")
    )
    .orderBy("TotalRevenue", ascending=False)
)

display(gold_top_products)

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 35128a49-4959-491b-834c-d6e01869f270)

In [40]:
gold_top_products.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_top_products")

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 42, Finished, Available, Finished, False)

In [41]:
from pyspark.sql.functions import sum

gold_top_customers = (
    gold_sales
    .groupBy("CustomerID")
    .agg(
        sum("Revenue").alias("TotalRevenue")
    )
    .orderBy("TotalRevenue", ascending=False)
)

display(gold_top_customers)

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 22cf5af2-7053-4d77-8e57-603d688c1cc0)

In [42]:
gold_top_customers.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_top_customers")

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 44, Finished, Available, Finished, False)

In [43]:
from pyspark.sql.functions import sum

gold_sales_channel = (
    gold_target_vs_actual
    .groupBy("SalesChannel")
    .agg(
        sum("ActualRevenue").alias("ActualRevenue"),
        sum("RevenueTarget").alias("RevenueTarget")
    )
)

display(gold_sales_channel)

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d71a63bb-886d-4a56-a3ac-32900084c81f)

In [44]:
from pyspark.sql.functions import sum

gold_sales_channel = (
    gold_target_vs_actual
    .groupBy("SalesChannel")
    .agg(
        sum("ActualRevenue").alias("ActualRevenue"),
        sum("RevenueTarget").alias("RevenueTarget")
    )
)

display(gold_sales_channel)

StatementMeta(, 9eb92471-6dd1-41c9-88cc-865f24b910c0, 46, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4d7a73d9-8b94-4a00-9844-9bdb11e9d191)